# HW1 – Recommender Systems Simulation
### Domain: Video Games (PC & PlayStation) — AUEB MSc Data Science

---

## Overview

This notebook simulates a synthetic video game rating dataset and uses **LDA (Latent Dirichlet Allocation)** to automatically discover user segments from their rating patterns — without ever directly observing which segment a user belongs to.

### Pipeline

```
game_titles.csv  (upload below)
      │
      ▼
generate_entities()  →  300 games  (title, platform, genre, price, Metacritic, …)
      │
      ▼
generate_users()     →  1 000 users  (5 segments × 200 users each)
      │
      ▼
generate_ratings()   →  10 000 binary ratings (+1 like / −1 dislike)  →  ratings.csv
      │
      ▼
learn_segments()     →  LDA topics  →  confusion matrix vs true segments
```

### The 5 User Segments

| # | Segment | Like condition |
|---|---|---|
| 1 | **PC Gamer** | platform ∈ {PC, BOTH} and Metacritic ≥ 60 |
| 2 | **Console Gamer** | platform ∈ {PS, BOTH} and Metacritic ≥ 55 |
| 3 | **Cross-Platform Gamer** | platform = BOTH and Metacritic ≥ 45 |
| 4 | **Budget Gamer** | price ≤ €25 |
| 5 | **Casual / Family Gamer** | PEGI age rating ∈ {3, 7} |

> Ratings are flipped with 10% probability (`noise=0.10`) to simulate real-world inconsistency.

In [ ]:
!pip install tomotopy -q

---
## Upload `game_titles.csv`

Run the cell below and select the `game_titles.csv` file from your computer.
This file contains the pool of ~560 real game titles used to generate entities.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select game_titles.csv from your computer

---
## Imports & Setup

In [ ]:
import random
import csv
import re
import numpy as np
import pandas as pd
import tomotopy as tp
from dataclasses import dataclass
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

SEG_NAMES = {
    1: 'PC Gamer',
    2: 'Console Gamer',
    3: 'Cross-Platform Gamer',
    4: 'Budget Gamer',
    5: 'Casual / Family Gamer'
}

# Load game titles from the uploaded CSV
_titles_df = pd.read_csv('game_titles.csv')
GAME_TITLES = _titles_df['title'].dropna().str.strip().tolist()
print(f'Loaded {len(GAME_TITLES)} game titles from game_titles.csv')


def _title_to_token(title: str) -> str:
    """Converts a game title to a safe LDA token (no spaces or special chars)."""
    safe = re.sub(r'[^A-Za-z0-9]+', '_', title)
    return safe.strip('_')

---
## Step 1 — `generate_entities()`

Generates **300 video games** by randomly sampling real titles from `game_titles.csv` and assigning synthetic attributes drawn from Gaussian distributions.

Each `VideoGame` object has **11 attributes**:

| Attribute | Type | Generation |
|---|---|---|
| `title` | str | sampled from `game_titles.csv` |
| `token` | str | URL-safe version of title for LDA (e.g. `Elden_Ring`) |
| `platform` | PC / PS / BOTH | sampled with weights 35% / 30% / 35% |
| `genre` | str | uniform over 10 genres |
| `price_eur` | float | Gaussian(35, 18), clipped [5, 80] |
| `metacritic` | int | Gaussian(68, 15), clipped [0, 100] |
| `avg_playtime_h` | float | Gaussian(25, 20), clipped [1, 200] |
| `is_multiplayer` | bool | True with prob 0.45 |
| `is_exclusive` | bool | True when platform ≠ BOTH |
| `age_rating` | PEGI | uniform over {3, 7, 12, 16, 18} |
| `release_year` | int | uniform [1994, 2024] |

In [ ]:
@dataclass
class VideoGame:
    title: str             # display name (e.g. "Elden Ring")
    token: str             # safe LDA token (e.g. "Elden_Ring")
    platform: str          # 'PC', 'PS', or 'BOTH'
    genre: str
    price_eur: float
    metacritic: int        # 0-100
    avg_playtime_h: float
    is_multiplayer: bool
    is_exclusive: bool     # True when platform != 'BOTH'
    age_rating: str        # PEGI: '3','7','12','16','18'
    release_year: int

In [ ]:
def generate_entities(
    game_num: int = 300,
    genre_options: List[str] = ['Action', 'RPG', 'Sports', 'Strategy', 'Horror',
                                 'Adventure', 'Simulation', 'Fighting', 'Puzzle', 'Racing'],
    platform_options: List[str] = ['PC', 'PS', 'BOTH'],
    platform_distro: List[float] = [0.35, 0.30, 0.35],
    price_gaussian_params: Tuple[float, float] = (35, 18),
    metacritic_gaussian_params: Tuple[float, float] = (68, 15),
    playtime_gaussian_params: Tuple[float, float] = (25, 20),
    multiplayer_prob: float = 0.45,
    age_ratings: List[str] = ['3', '7', '12', '16', '18'],
    year_range: Tuple[int, int] = (1994, 2024)
) -> List[VideoGame]:
    """
    Generates a list of synthetic video game entities sampled from GAME_TITLES.

    Args:
        game_num (int): Number of games. Default: 300.

    Returns:
        List[VideoGame]: Generated game objects.
    """
    titles = random.sample(GAME_TITLES, min(game_num, len(GAME_TITLES)))
    games = []
    for title in titles:
        platform_idx = np.random.choice(len(platform_options), p=platform_distro)
        platform     = platform_options[platform_idx]
        genre        = random.choice(genre_options)
        price        = round(float(np.clip(random.gauss(*price_gaussian_params), 5.0, 80.0)), 2)
        metacritic   = int(np.clip(random.gauss(*metacritic_gaussian_params), 0, 100))
        playtime     = round(float(np.clip(random.gauss(*playtime_gaussian_params), 1.0, 200.0)), 1)
        is_multi     = random.random() < multiplayer_prob
        is_excl      = platform != 'BOTH'
        age_rating   = random.choice(age_ratings)
        year         = random.randint(*year_range)
        games.append(VideoGame(
            title=title, token=_title_to_token(title),
            platform=platform, genre=genre,
            price_eur=price, metacritic=metacritic, avg_playtime_h=playtime,
            is_multiplayer=is_multi, is_exclusive=is_excl,
            age_rating=age_rating, release_year=year
        ))
    return games

In [ ]:
games = generate_entities(game_num=300)
print(f'Generated {len(games)} games.')
print(vars(games[0]))
print('Platform distribution:', pd.Series([g.platform for g in games]).value_counts().to_dict())

---
## Step 2 — `generate_users()`

Generates **1 000 users** split equally across 5 segments (200 per segment). Each segment has a distinct age profile to reflect realistic demographics. Users are shuffled after creation so segment order is not preserved.

| Segment | Age distribution |
|---|---|
| PC Gamer | Gaussian(30, 6) |
| Console Gamer | Gaussian(25, 7) |
| Cross-Platform Gamer | Gaussian(22, 5) |
| Budget Gamer | Gaussian(20, 8) |
| Casual / Family Gamer | Gaussian(38, 10) |

In [ ]:
@dataclass
class User:
    segment: int
    age: int
    gender: str

In [ ]:
def generate_users_segment1(user_num: int = 200) -> List[User]:
    """Segment 1 - PC Gamer. Age: Gaussian(30, 6)."""
    return [User(segment=1, age=max(10, int(random.gauss(30, 6))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]


def generate_users_segment2(user_num: int = 200) -> List[User]:
    """Segment 2 - Console Gamer. Age: Gaussian(25, 7)."""
    return [User(segment=2, age=max(10, int(random.gauss(25, 7))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]


def generate_users_segment3(user_num: int = 200) -> List[User]:
    """Segment 3 - Cross-Platform Gamer. Age: Gaussian(22, 5)."""
    return [User(segment=3, age=max(10, int(random.gauss(22, 5))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]


def generate_users_segment4(user_num: int = 200) -> List[User]:
    """Segment 4 - Budget Gamer. Age: Gaussian(20, 8)."""
    return [User(segment=4, age=max(10, int(random.gauss(20, 8))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]


def generate_users_segment5(user_num: int = 200) -> List[User]:
    """Segment 5 - Casual / Family Gamer. Age: Gaussian(38, 10)."""
    return [User(segment=5, age=max(10, int(random.gauss(38, 10))),
                 gender=random.choice(['M', 'F'])) for _ in range(user_num)]

In [ ]:
def generate_users(user_num: int = 1000) -> List[User]:
    """
    Generates a shuffled population of users split equally across 5 segments.

    Args:
        user_num (int): Total users (user_num // 5 per segment). Default: 1000.

    Returns:
        List[User]: Shuffled User objects tagged with segment id (1-5).
    """
    n = user_num // 5
    users = (
        generate_users_segment1(n) +
        generate_users_segment2(n) +
        generate_users_segment3(n) +
        generate_users_segment4(n) +
        generate_users_segment5(n)
    )
    random.shuffle(users)
    return users

In [ ]:
users = generate_users(user_num=1000)
counts = pd.Series([u.segment for u in users]).value_counts().sort_index()
for seg, cnt in counts.items():
    print(f'  Segment {seg} - {SEG_NAMES[seg]:25s}: {cnt} users')

---
## Step 3 — `generate_ratings()`

Generates **10 000 (user, game) pairs** and assigns a binary rating (+1 like / −1 dislike) based on each segment's like condition. Each rating is then independently flipped with probability `noise=0.10`.

A separate helper function handles each segment:

```
generate_ratings_segment1()  →  PC Gamer:           PC/BOTH + Metacritic ≥ 60
generate_ratings_segment2()  →  Console Gamer:      PS/BOTH + Metacritic ≥ 55
generate_ratings_segment3()  →  Cross-Platform:     BOTH    + Metacritic ≥ 45
generate_ratings_segment4()  →  Budget Gamer:       price ≤ €25
generate_ratings_segment5()  →  Casual/Family:      PEGI ∈ {3, 7}
```

Results are saved to `ratings.csv`.

In [ ]:
def _make_row(user, game, rating, reason):
    """Helper: flattens a user-game pair into a CSV row dict."""
    return {'segment': user.segment, 'age': user.age, 'gender': user.gender,
            'game': game.token, 'platform': game.platform, 'genre': game.genre,
            'price_eur': game.price_eur, 'metacritic': game.metacritic,
            'avg_playtime_h': game.avg_playtime_h, 'is_multiplayer': game.is_multiplayer,
            'is_exclusive': game.is_exclusive, 'age_rating': game.age_rating,
            'release_year': game.release_year, 'rating': rating, 'reason': reason}

In [ ]:
def generate_ratings_segment1(users, games, pairs, noise):
    """Segment 1 - PC Gamer. Like if platform in ('PC', 'BOTH') AND Metacritic >= 60."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform not in ('PC', 'BOTH'):
            rating, reason = -1, 'Not on PC'
        elif game.metacritic >= 60:
            rating, reason = 1, 'PC/BOTH + good Metacritic'
        else:
            rating, reason = -1, 'PC but Metacritic too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment2(users, games, pairs, noise):
    """Segment 2 - Console Gamer. Like if platform in ('PS', 'BOTH') AND Metacritic >= 55."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform not in ('PS', 'BOTH'):
            rating, reason = -1, 'Not on PlayStation'
        elif game.metacritic >= 55:
            rating, reason = 1, 'PS/BOTH + acceptable Metacritic'
        else:
            rating, reason = -1, 'PS platform but Metacritic too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment3(users, games, pairs, noise):
    """Segment 3 - Cross-Platform Gamer. Like if platform == 'BOTH' AND Metacritic >= 45."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.platform != 'BOTH':
            rating, reason = -1, 'Not on both platforms'
        elif game.metacritic >= 45:
            rating, reason = 1, 'BOTH + acceptable Metacritic'
        else:
            rating, reason = -1, 'BOTH but Metacritic too low'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment4(users, games, pairs, noise):
    """Segment 4 - Budget Gamer. Like if price_eur <= 25."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.price_eur <= 25:
            rating, reason = 1, 'Cheap enough'
        else:
            rating, reason = -1, 'Too expensive'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows


def generate_ratings_segment5(users, games, pairs, noise):
    """Segment 5 - Casual / Family Gamer. Like if age_rating in ('3', '7')."""
    rows = []
    for u_idx, g_idx in pairs:
        user, game = users[u_idx], games[g_idx]
        if game.age_rating in ('3', '7'):
            rating, reason = 1, 'Family age rating'
        else:
            rating, reason = -1, 'Age rating too high'
        if random.random() < noise:
            rating *= -1; reason += ' [NOISE FLIP]'
        rows.append(_make_row(user, game, rating, reason))
    return rows

In [ ]:
def generate_ratings(
    users: List[User],
    games: List[VideoGame],
    n_ratings: int = 10000,
    noise: float = 0.10,
    output_file: str = 'ratings.csv'
) -> None:
    """
    Generates binary ratings and writes them to CSV.

    Args:
        users, games: from generate_users() and generate_entities().
        n_ratings (int): Number of (user, game) pairs. Default: 10000.
        noise (float): Rating flip probability. Default: 0.10.
        output_file (str): CSV output path.
    """
    all_pairs = [(u, g) for u in range(len(users)) for g in range(len(games))]
    sampled   = random.sample(all_pairs, min(n_ratings, len(all_pairs)))

    seg_pairs = {s: [] for s in range(1, 6)}
    for u_idx, g_idx in sampled:
        seg_pairs[users[u_idx].segment].append((u_idx, g_idx))

    handlers = {
        1: generate_ratings_segment1,
        2: generate_ratings_segment2,
        3: generate_ratings_segment3,
        4: generate_ratings_segment4,
        5: generate_ratings_segment5,
    }

    all_rows = []
    for s in range(1, 6):
        all_rows.extend(handlers[s](users, games, seg_pairs[s], noise))
    random.shuffle(all_rows)

    fieldnames = ['segment', 'age', 'gender', 'game', 'platform', 'genre', 'price_eur',
                  'metacritic', 'avg_playtime_h', 'is_multiplayer', 'is_exclusive',
                  'age_rating', 'release_year', 'rating', 'reason']
    with open(output_file, 'w', newline='', encoding='utf-8') as fw:
        writer = csv.DictWriter(fw, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    total    = len(all_rows)
    positive = sum(1 for r in all_rows if r['rating'] == 1)
    print(f'Total ratings     : {total:,}')
    print(f'Positive (like)   : {positive:,}  ({positive/total:.1%})')
    print(f'Negative (dislike): {total-positive:,}  ({(total-positive)/total:.1%})')
    print(f'Noise level       : {noise:.0%}')
    print(f'Saved to          : {output_file}')

In [ ]:
generate_ratings(users, games, n_ratings=10000, noise=0.10, output_file='ratings.csv')

df = pd.read_csv('ratings.csv')
print()
print('Like rate per segment:')
for seg, grp in df.groupby('segment'):
    print(f'  Segment {seg} - {SEG_NAMES[seg]:25s}: {(grp["rating"]==1).mean():.1%} likes  ({len(grp):,} ratings)')

In [ ]:
df.head(10)

---
## Step 4 — `learn_segments()` — LDA with Anchor Words

This step treats each user as a **document** and each rated game interaction as a **token**, then trains LDA to discover latent topics that correspond to the 5 segments.

### Token format
Each rating becomes a string token: `<game_token>_<platform>_<genre>_LIKE` or `..._DISLIKE`

> Example: `Elden_Ring_PC_RPG_LIKE`

### Anchor words
To guide LDA toward the expected segments, we seed each topic with up to 20 **anchor words** — tokens we know are characteristic of that segment:

| Topic | Anchor type | Target segment |
|---|---|---|
| 0 | tokens containing `_PC_` | PC Gamer |
| 1 | tokens containing `_PS_` | Console Gamer |
| 2 | tokens containing `_BOTH_` | Cross-Platform Gamer |
| 3 | LIKE tokens for cheap games (price ≤ €25) | Budget Gamer |
| 4 | LIKE tokens for family-rated games (PEGI 3/7) | Casual / Family Gamer |

After training, the most-probable topic for each user is compared against their true segment in the **confusion matrix**.

In [ ]:
def learn_segments(
    ratings_file: str = 'ratings.csv',
    games: List[VideoGame] = None,
    k: int = 5,
    n_iter: int = 500,
    top_n_words: int = 10
) -> pd.DataFrame:
    """
    Discovers customer segments from rating patterns using LDA (tomotopy).

    Args:
        ratings_file (str): Path to the CSV from generate_ratings().
        games (List[VideoGame]): Game objects for anchor word lookup.
        k (int): Number of LDA topics. Default: 5.
        n_iter (int): Training iterations. Default: 500.
        top_n_words (int): Top words to show per topic. Default: 10.

    Returns:
        pd.DataFrame: Cross-tab of true segment vs most-probable LDA topic.
    """
    df = pd.read_csv(ratings_file)
    game_lookup = {g.token: g for g in games} if games else {}

    # Build one document per user
    df['_uid'] = df['segment'].astype(str) + '_' + df['age'].astype(str) + '_' + df['gender']
    user_docs = {}
    for uid, grp in df.groupby('_uid'):
        tokens = [
            f"{r['game']}_{r['platform']}_{r['genre']}_{'LIKE' if r['rating']==1 else 'DISLIKE'}"
            for _, r in grp.iterrows()
        ]
        if tokens:
            user_docs[uid] = tokens

    print(f'Users (documents): {len(user_docs)}  |  Avg tokens/user: {np.mean([len(v) for v in user_docs.values()]):.1f}')

    # Build anchor word lists (up to 20 per topic)
    all_tokens = set(t for doc in user_docs.values() for t in doc)
    anchor_pc   = [t for t in all_tokens if '_PC_' in t][:20]
    anchor_ps   = [t for t in all_tokens if '_PS_' in t][:20]
    anchor_both = [t for t in all_tokens if '_BOTH_' in t][:20]
    anchor_budget, anchor_casual = [], []
    if game_lookup:
        for t in all_tokens:
            g_token = t.rsplit('_', 3)[0]
            if g_token not in game_lookup:
                continue
            g = game_lookup[g_token]
            if g.price_eur <= 25 and t.endswith('LIKE'):
                anchor_budget.append(t)
            if g.age_rating in ('3', '7') and t.endswith('LIKE'):
                anchor_casual.append(t)
    anchor_budget = anchor_budget[:20]
    anchor_casual = anchor_casual[:20]

    print(f'Anchor words  →  PC: {len(anchor_pc)}, PS: {len(anchor_ps)}, '
          f'BOTH: {len(anchor_both)}, Budget: {len(anchor_budget)}, Casual: {len(anchor_casual)}')

    # Train LDA
    lda = tp.LDAModel(k=k, seed=42)
    for doc_tokens in user_docs.values():
        lda.add_doc(doc_tokens)

    print(f'Training LDA  ({n_iter} iterations)...')
    lda.train(n_iter)
    print(f'Done  |  log-likelihood per word: {lda.ll_per_word:.4f}')

    # Topic summaries
    print()
    for tid in range(lda.k):
        top_words = [p[0] for p in lda.get_topic_words(tid, top_n=top_n_words)]
        pc_c  = sum(1 for w in top_words if '_PC_' in w)
        ps_c  = sum(1 for w in top_words if '_PS_' in w)
        bot_c = sum(1 for w in top_words if '_BOTH_' in w)
        like_c = sum(1 for w in top_words if w.endswith('_LIKE'))
        dominant = max({'PC': pc_c, 'PS': ps_c, 'BOTH': bot_c},
                       key=lambda x: {'PC': pc_c, 'PS': ps_c, 'BOTH': bot_c}[x])
        genres = [p[2] for w in top_words if len(p := w.rsplit('_', 3)) == 4]
        top_genre = pd.Series(genres).value_counts().index[0] if genres else '?'
        print(f'Topic {tid}  |  platform: {dominant:4s}  genre: {top_genre:12s}  '
              f'LIKE: {like_c}/{top_n_words}  top: {top_words[0]}')

    # Cross-tab: true segment vs most-probable LDA topic
    uid_list = list(user_docs.keys())
    rows = [
        {'true_segment': int(uid_list[i].split('_')[0]),
         'lda_topic': int(np.argmax(doc.get_topic_dist()))}
        for i, doc in enumerate(lda.docs)
    ]
    ct = pd.crosstab(
        pd.DataFrame(rows)['true_segment'],
        pd.DataFrame(rows)['lda_topic'],
        rownames=['True Segment'], colnames=['LDA Topic']
    )
    return ct

In [ ]:
ct = learn_segments(ratings_file='ratings.csv', games=games, k=5, n_iter=500)

---
## Results — Confusion Matrix

Each row is a **true segment**, each column is the **LDA topic** most users from that segment were assigned to.
A good result shows one dominant topic per segment (diagonal-like pattern).

In [ ]:
ct_full = ct.reindex(index=range(1, 6), columns=range(5), fill_value=0)

row_labels = [f'Seg {i} – {SEG_NAMES[i]}' for i in range(1, 6)]
col_labels  = [f'Topic {j}' for j in range(5)]
cm_df = pd.DataFrame(ct_full.values, index=row_labels, columns=col_labels)

print('Confusion Matrix  (rows = true segment, columns = LDA topic)\n')
print(cm_df.to_string())
print()
for i, row in enumerate(ct_full.values):
    total = row.sum(); best = row.max(); seg = i + 1
    print(f'  Seg {seg} – {SEG_NAMES[seg]:25s}: {best}/{total} ({best/total:.0%}) → Topic {row.argmax()}')